In [ ]:
import pandas as pd
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
from config.constants import PROJECT_ROOT
import numpy as np
import pickle

%load_ext autoreload
%autoreload 2

### Load constants

In [ ]:
mimic_top_100_path = "../data/top_features/mimic_top100_features.pkl"
with open(mimic_top_100_path, "rb") as f:
    mimic_top_100 = list(map(str, pickle.load(f)))#
    
edge_cc =  pd.read_csv("../saved_data/cohorts/DTB/new/selected_edges_DTB_all.csv")
edge_cc["cohort_name"] = edge_cc["D1"] + "-" + edge_cc["D2"]
edge_cc = edge_cc[['cohort_name','D1', 'D2', 'RR','counts_cohort',
       'female_counts_cohort', 'AGE_AT_DISEASE_cohort',
       'CODE_DIFF_DAYS_cohort', 'death_counts_cohort']]

edge_cc["female_rate"] = edge_cc["female_counts_cohort"] / edge_cc["counts_cohort"]
edge_cc["death_rate"] = edge_cc["death_counts_cohort"] / edge_cc["counts_cohort"]
edge_cc = edge_cc.rename(columns={
    "AGE_AT_DISEASE_cohort": "age",
    "CODE_DIFF_DAYS_cohort": "W",
}).drop(columns=["female_counts_cohort", "death_counts_cohort"])

### select rep cohorts

In [ ]:
rep_name = "representative_cohorts_K100_ab4b58.txt"
rep_cohorts = pd.read_csv(f"../saved_data/hpc_run_3008/rep_selections/{rep_name}",header=None,names=["cohort_name"])

In [ ]:
#selected_cohorts = rep_cohorts.iloc[[10,15,67,55,99]] #,0,5,76,43,23]] #random#
#selected_cohorts = rep_cohorts.iloc[[0,5,76,43,23]] 
selected_cohorts = rep_cohorts.iloc[[0,5,76,43,23]] #random#

In [ ]:
RESULTS_DIR = Path(PROJECT_ROOT) / "saved_data" / "hpc_run_3008" / "results"
METRICS = ["auroc", "auprc", "f1", "mcc", "recall", "precision"]

def load_results(cohorts, prefix, agg,variant, models=("random_forest",), folds=(0,),results_dir=RESULTS_DIR):
    rows, missing = [], []
    for cohort in cohorts:
        for model in models:
            for f in folds:
                file = results_dir / cohort / model / prefix / f"fold_{f}" / f"agg_int_{str(agg)}" / "impute_fill" / f"variant_{variant}" / "results_final.csv"
                if not file.exists():
                    missing.append(file)
                    print(missing)
                    continue
                    
                rows.append(pd.read_csv(file).assign(
                    cohort_name=cohort, model=model, prefix=prefix, fold=f))
    if missing:
        print(f"{len(missing)} missing, e.g. {missing[:3]}")
    if not rows:
        return pd.DataFrame()
    df = pd.concat(rows, ignore_index=True)
    df[METRICS] = df[METRICS].apply(pd.to_numeric, errors="coerce")
    return df[["cohort_name"] + METRICS]


In [ ]:
def load_folds(cohort, fold=0, seed=42, first_adm_only=True):
    suffix = "_firstadm" if first_adm_only else ""
    f = Path(PROJECT_ROOT) / "saved_data" / "folds" / cohort / f"seed_{seed}{suffix}" / f"fold_{fold}.pkl"
    if not f.exists():
        return None#
    with open(f, "rb") as fh:
        train, val, test = pickle.load(fh)
    ids = {"train": train[:, 1], "val": val[:, 1], "test": test[:, 1]}
    
    return ids

def load_cohort(cohort, fold=0, seed=42):
    df = pd.read_csv(Path(PROJECT_ROOT) / "saved_data" / "cohorts" / "DTB" / "new" / f"{cohort}.csv.gz",compression="gzip")
    return df[['subject_id', 'hadm_id', 'admittime', 'dischtime','gender', 'age','dod','D1_date', 'D1_W', 'D1_5y','D2_date', 'label']]

def load_features(cohort, days=14):
    df = pd.read_csv(Path(PROJECT_ROOT) / "saved_data" / f"features_{days}d" / cohort / "features.csv.gz",
                       dtype={"subject_id": "int64", "hadm_id": "int64", "minute": "int64",
                              "itemid": "string", "value": "float64"})
    return df

add cohort cc 

In [ ]:
rows = []
for cohort in rep_cohorts["cohort_name"]:
    cohort_df = load_cohort(cohort)
    cohort_first_df = cohort_df.sort_values("admittime", kind="stable").drop_duplicates("subject_id", keep="first")
    #cohort_first_df = cohort_first_df[cohort_first_df.hadm_id.isin(train_val_ids)]
    n = len(cohort_first_df)
    n_positive = int(cohort_first_df["label"].sum())
    rows.append({
        "cohort_name": cohort,
        "n": n,
        "n_positive": n_positive,
        "target_rate": n_positive / n if n else float("nan"),
    })

cohort_cc = pd.DataFrame(rows)

# merge with edge info
cohort_cc = cohort_cc.merge(edge_cc, on = "cohort_name")


In [ ]:
cohort_cc[cohort_cc.cohort_name.isin(selected_cohorts.cohort_name)]# without removing the features #ToDo 

---


In [ ]:
cohort_name = "C25-K59"

----

In [ ]:
ids =  load_folds(cohort_name)

train_val_ids = np.concatenate([ids["train"], ids["val"]])

feat_df = load_features(cohort_name)
feat_df = feat_df[feat_df.hadm_id.isin(train_val_ids)]
feat_df = feat_df[feat_df.itemid.isin(mimic_top_100)]




In [ ]:
fold_ids = set(train_val_ids)
feat_ids = set(feat_df.hadm_id.unique())
cohort_ids = set(cohort_first_df.hadm_id.unique())

print("fold == feat  :", fold_ids == feat_ids, len(fold_ids),len(feat_ids))
print("fold == cohort:", fold_ids == cohort_ids, len(fold_ids),len(cohort_ids))
print("feat == cohort:", feat_ids == cohort_ids, len(feat_ids),len(cohort_ids))

print("\nin fold not in feat   :", fold_ids - feat_ids)
print("in feat not in fold   :", feat_ids - fold_ids)
print("in fold not in cohort :", fold_ids - cohort_ids)
print("in cohort not in fold :", cohort_ids - fold_ids)
print("in feat not in cohort :", feat_ids - cohort_ids)
print("in cohort not in feat :", cohort_ids - feat_ids)


In [ ]:
len(ids["train"]) + len(ids["val"]) + len(ids["test"])

# Check results

In [ ]:
hpc_run_dir = Path(PROJECT_ROOT) / "saved_data" / "hpc_run_3008" / "results"
prefix = "first_adms_14d_mimic100"
agg = 24
variant = "vmd"
hpc_run_results = load_results(rep_cohorts["cohort_name"], prefix, agg, "VMD", results_dir=hpc_run_dir)


In [ ]:
selected_cohorts

## ML models new Runs

In [ ]:
COHORTS = selected_cohorts.cohort_name
COHORTS.tolist()

COHORTSSINA = ["F10-K63","M25-D47", "M48-M96"]

In [ ]:
selected_cohorts = rep_cohorts.iloc[[10,15,67,55,99]]
selected_cohorts

In [ ]:
import argparse
from pathlib import Path
from flab_training.envmanager import EnvManager
from config.constants import PROJECT_ROOT

#selected_cohorts = rep_cohorts.iloc[[10,15,67,55,99]]

COHORTS = ['C64-C78'] #COHORTSSINA #['C64-C78', 'K44-K59', 'I12-K65', 'E78-G81']#selected_cohorts.cohort_name

for cohort in COHORTS:
    print(f"\n=== {cohort} ===")
    args = argparse.Namespace(
        days_before_discharge=180,
        dataset="MIMIC_IV",
        cohort=cohort,
        fold=0,
        model_type="lstm",
        grid="nested",
        pretrain=False,
        feature_selection=True,
        feature_selection_method="mimic_top_100",
        load_ckpt_path=None,
        prefix="test_3",
        static_threshold=0,
        hid_dim_demo=64,
        agg_int=1440, # hours 
        drop_minutes=False,
        freeze=False,
        config_path=str(Path(PROJECT_ROOT) / "flab_training" / "config_params.yaml"),
        split_seed=None,
        fast=False,
        search="grid",
        n_trials=25,
        impute="fill",
        variant="stats",
        extractor="DTB",
        first_adm_only=False,
        oversampling="minority",
        agg = "mean",
    )
    EnvManager(args).train_full()


In [ ]:
main_dir = Path(PROJECT_ROOT) / "saved_data" / "results"

In [ ]:
COHORTS = selected_cohorts.cohort_name
COHORTS = COHORTSSINA

In [ ]:
main_dir = Path(PROJECT_ROOT) / "saved_data" / "results"
prefix = "test_count"
agg = 1440
variant = "count"
for model in ["random_forest","catboost"]:
    new_results_stats = load_results(COHORTS, prefix, agg, variant, models=[model], results_dir=main_dir)
    print(new_results_stats)


In [ ]:
main_dir = Path(PROJECT_ROOT) / "saved_data" / "results"
prefixes = ["test","test_3"]
agg = 1440
variant = "stats"
for model in ["random_forest","lstm","gru"]:#,"catboost","xgboost"]:
    for prefix in prefixes:
        new_results_stats = load_results(COHORTS, prefix, agg, variant, models=[model], results_dir=main_dir)
        print(model,prefix)
        print(new_results_stats)


In [ ]:
main_dir = Path(PROJECT_ROOT) / "saved_data" / "results"
prefix = "test_14_daily"
agg = 24
variant = "VMD"
new_results_stats = load_results(COHORTS, prefix, agg, variant, models=["random_forest"], results_dir=main_dir)
new_results_stats

In [ ]:
main_dir = Path(PROJECT_ROOT) / "saved_data" / "results"
prefix = "first_adms_14d_mrmr100"
agg = 24
variant = "VMD"
new_results_stats = load_results(COHORTS, prefix, agg, variant, models=["random_forest"], results_dir=main_dir)
new_results_stats

In [ ]:
main_dir = Path(PROJECT_ROOT) / "saved_data" / "results"
prefix = "initial_run"
agg = 24
variant = "VMD"
new_results_stats = load_results(COHORTS, prefix, agg, variant, models=["random_forest"], results_dir=main_dir)
new_results_stats


In [ ]:
main_dir = Path(PROJECT_ROOT) / "saved_data" / "results"
prefix = "initial_run_180_d"
agg = 24
variant = "VMD"
new_results_stats = load_results(COHORTS, prefix, agg, variant, models=["random_forest"], results_dir=main_dir)
new_results_stats

In [ ]:
main_dir = Path(PROJECT_ROOT) / "saved_data" / "results"
agg = 1440
variant = "stats"
#prefixes  = ["stats_180_3bins","stats_180_3bins_cross_features","stats_180_3bins_new_processor","stats_180_3bins_new_processor_old_trim"]
prefixes  = ["stats_180_3bins_new_processor_old_trim","stats_180_3bins_new_processor_ohne_trim"]
for p in prefixes: 
    new_results_stats = load_results(COHORTS, p, agg, variant, models=["random_forest"], results_dir=main_dir)
    print(p ,new_results_stats)

# Plot

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

main_dir = Path(PROJECT_ROOT) / "saved_data" / "results"
METRICS = ["auroc", "auprc", "f1", "mcc", "recall", "precision"]

CONFIGS = {
    "180d_3bin_stats":         dict(prefix="stats_180_3bins_new_processor_old_trim",  agg=1440, variant="stats", days=180),
    "180d_3bin_stats_no_trim": dict(prefix="stats_180_3bins_new_processor_ohne_trim", agg=1440, variant="stats", days=180),
    "initial_14d_vmd":         dict(prefix="initial_run",                             agg=24,   variant="VMD",   days=14),
    "initial_180d_vmd":        dict(prefix="initial_run_180_d",                       agg=24,   variant="VMD",   days=180),
}
MODEL, FOLD = "random_forest", 0

COLORS = plt.cm.tab10(np.linspace(0, 1, 10))

def leaf_path(cohort, cfg, results_dir):
    return (results_dir / cohort / MODEL / cfg["prefix"] / f"fold_{FOLD}" /
            f"agg_int_{cfg['agg']}" / "impute_fill" / f"variant_{cfg['variant']}" / "results_final.csv")

def load_configs(cohorts, configs, results_dir):
    rows, missing = [], []
    for cohort in cohorts:
        for label, cfg in configs.items():
            f = leaf_path(cohort, cfg, results_dir)
            if not f.exists():
                missing.append(f)
                continue
            rows.append(pd.read_csv(f).assign(cohort_name=cohort, config=label, **cfg))
    if missing:
        print(f"{len(missing)} missing, e.g. {missing[0]}")
    if not rows:
        return pd.DataFrame()
    df = pd.concat(rows, ignore_index=True)
    df[METRICS] = df[METRICS].apply(pd.to_numeric, errors="coerce")
    return df[["cohort_name", "config", "prefix", "agg", "variant", "days"] + METRICS]

all_cohorts = COHORTSSINA #sorted(d.name for d in main_dir.iterdir() if d.is_dir() and d.name != "cohort_name")
COHORTS = [c for c in all_cohorts if all(leaf_path(c, cfg, main_dir).exists() for cfg in CONFIGS.values())]
print(f"{len(COHORTS)} cohorts with all {len(CONFIGS)} configs: {COHORTS}")

combined = load_configs(COHORTS, CONFIGS, main_dir)

wide = combined.pivot(index="cohort_name", columns="config", values="auroc")
wide = wide[list(CONFIGS)].dropna().sort_values(list(CONFIGS)[0])

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(wide))
for v, c in zip(CONFIGS, COLORS):
    ax.plot(x, wide[v], marker="o", ms=4, lw=1, color=c, label=v, alpha=0.85)

ax.set_xticks(x, wide.index, rotation=90, fontsize=6)
ax.axhline(0.5, color="#52514e", lw=1, ls=(0, (4, 3)))
ax.set_ylabel("AUROC")
ax.set_title("Per-cohort AUROC across configs")
ax.legend(frameon=False)
ax.grid(axis="y", lw=0.7, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path(PROJECT_ROOT) / "saved_data"
METRICS = ["auroc", "auprc", "f1", "mcc", "recall", "precision"]

'''COHORTS = ["C25-K59", "C64-C78", "K44-K59", "I12-K65", "E78-G81"]
MODEL, FOLD = "random_forest", 0

CONFIGS = {
    "VMD 180d":       dict(root="results_trials",      prefix="test_3108_stats_180_3bin_ov", agg=24,   variant="VMD"),
    "STATS 180d":     dict(root="results_trials",      prefix="stats_180d_3bin_ov",          agg=1440, variant="STATS"),
    "stats new proc": dict(root="results",             prefix="stats_180_3bins_new_processor_old_trim", agg=1440, variant="stats"),
    "VMD 14d":        dict(root="hpc_run_3008/results", prefix="first_adms_14d_mimic100",    agg=24,   variant="VMD"),
}'''

COHORTS = ["C91-D63", "E05-E89", "J45-J38", "I73-J81", "O26-O34"]

CONFIGS = {
    "180d mimic100":  dict(root="results_trials",       prefix="test_3108_stats_180_3bin_ov",          agg=24, variant="VMD"),
    "180d newfeat":   dict(root="results_trials",       prefix="test_3108_stats_180_3bins_ov_newfeat", agg=24, variant="VMD"),
    "360d 4bin":      dict(root="results_trials",       prefix="test_3108_stats_360_4bin_ov",          agg=24, variant="VMD"),
    "14d mimic100":   dict(root="hpc_run_3008/results", prefix="first_adms_14d_mimic100",              agg=24, variant="VMD"),
    "14d mrmr100":    dict(root="hpc_run_3008/results", prefix="first_adms_14d_mrmr100",               agg=24, variant="VMD"),
}


def leaf_path(cohort, cfg):
    return (ROOT / cfg["root"] / cohort / MODEL / cfg["prefix"] / f"fold_{FOLD}" /
            f"agg_int_{cfg['agg']}" / "impute_fill" / f"variant_{cfg['variant']}" / "results_final.csv")

rows, missing = [], []
for cohort in COHORTS:
    for label, cfg in CONFIGS.items():
        f = leaf_path(cohort, cfg)
        if not f.exists():
            missing.append(f)
            continue
        rows.append(pd.read_csv(f).assign(cohort_name=cohort, config=label))
if missing:
    print(f"{len(missing)} missing, e.g. {missing[0]}")

combined = pd.concat(rows, ignore_index=True)
combined[METRICS] = combined[METRICS].apply(pd.to_numeric, errors="coerce")

wide = combined.pivot(index="cohort_name", columns="config", values="auroc")
wide = wide[list(CONFIGS)].dropna().sort_values(list(CONFIGS)[0])

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(wide))
for v, c in zip(CONFIGS, plt.cm.tab10.colors):
    ax.plot(x, wide[v], marker="o", ms=6, lw=1.2, color=c, label=v, alpha=0.85)

ax.set_xticks(x, wide.index, rotation=45, ha="right")
ax.axhline(0.5, color="#52514e", lw=1, ls=(0, (4, 3)))
ax.set_ylabel("AUROC")
ax.set_title("Per-cohort AUROC across variants")
ax.legend(frameon=False)
ax.grid(axis="y", lw=0.7, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path(PROJECT_ROOT) / "saved_data"
METRICS = ["auroc", "auprc", "f1", "mcc", "recall", "precision"]
MODEL, FOLD = "random_forest", 0

COHORTS = ["C25-K59", "C64-C78", "K44-K59", "I12-K65", "E78-G81",
           "C91-D63", "E05-E89", "J45-J38", "I73-J81", "O26-O34"]

CONFIGS = {
    "180d stats": dict(root="results_trials",       prefix="test_3108_stats_180_3bin_ov",          agg=24, variant="VMD"),
    #"180d newfeat":  dict(root="results_trials",       prefix="test_3108_stats_180_3bins_ov_newfeat", agg=24, variant="VMD"),
    "14d mimic100":  dict(root="hpc_run_3008/results", prefix="first_adms_14d_mimic100",              agg=24, variant="VMD"),
    "14d mrmr100":   dict(root="hpc_run_3008/results", prefix="first_adms_14d_mrmr100",               agg=24, variant="VMD"),
}

def leaf_path(cohort, cfg):
    return (ROOT / cfg["root"] / cohort / MODEL / cfg["prefix"] / f"fold_{FOLD}" /
            f"agg_int_{cfg['agg']}" / "impute_fill" / f"variant_{cfg['variant']}" / "results_final.csv")

rows, missing = [], []
for cohort in COHORTS:
    for label, cfg in CONFIGS.items():
        f = leaf_path(cohort, cfg)
        if not f.exists():
            missing.append(f)
            continue
        rows.append(pd.read_csv(f).assign(cohort_name=cohort, config=label))
if missing:
    print(f"{len(missing)} missing, e.g. {missing[0]}")

combined = pd.concat(rows, ignore_index=True)
combined[METRICS] = combined[METRICS].apply(pd.to_numeric, errors="coerce")

for metric in ["auroc"]:
    wide = combined.pivot(index="cohort_name", columns="config", values=metric)
    wide = wide.reindex(index=COHORTS, columns=list(CONFIGS))
    wide.loc["MEAN"] = wide.mean()
    print(f"\n=== {metric.upper()} ===")
    print(wide.round(3).to_string())


In [ ]:
wide

In [ ]:
cohort_cc[cohort_cc.cohort_name.isin(COHORTS)].merge(wide,on="cohort_name")